# Hybrid+Both Pipeline

This notebook builds the canonical hybrid+both bundle, optionally warms the xTB cache, runs a lightweight smoke test, and then launches training.

The intended path is:
1. Build bundle from the clean base dataset + train-only XenoSite auxiliary.
2. Optionally precompute xTB for the union set.
3. Smoke test the live LNN + wave + analogical bridge.
4. Train with the full-xTB hybrid trainer.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
from pathlib import Path

repo_dir = Path('/content/enzyme_Software')
git_url = 'https://github.com/Nikku03/enzyme_Software.git'
%cd /content
if not repo_dir.exists():
    subprocess.check_call(['git', 'clone', git_url, str(repo_dir)])
%cd /content/enzyme_Software
subprocess.check_call(['git', 'pull', 'origin', 'main'])

repo = '/content/enzyme_Software'
src = '/content/enzyme_Software/src'
if repo not in sys.path:
    sys.path.insert(0, repo)
if src not in sys.path:
    sys.path.insert(0, src)

need_stack = False
try:
    from rdkit import Chem  # noqa: F401
except Exception:
    need_stack = True
try:
    from sklearn.metrics import roc_auc_score  # noqa: F401
except Exception:
    need_stack = True
if need_stack:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'numpy<2', 'scipy', 'scikit-learn', 'rdkit-pypi'
    ])

!python -V
!git log --oneline -5


In [ ]:
os.environ['HYBRID_BOTH_BUNDLE'] = 'data/hybrid_both_bundle/bundle.json'
os.environ['HYBRID_BOTH_XTB_CACHE'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb_hybrid_both'
os.environ['HYBRID_BOTH_OUTPUT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_both'
os.environ['HYBRID_BOTH_ARTIFACT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_both'
os.environ['HYBRID_BOTH_EPOCHS'] = '75'
os.environ['HYBRID_BOTH_BATCH_SIZE'] = '16'
os.environ['HYBRID_BOTH_LR'] = '2e-4'
os.environ['HYBRID_BOTH_WD'] = '1e-4'
os.environ['HYBRID_BOTH_LIMIT'] = '0'
os.environ['HYBRID_BOTH_COMPUTE_XTB_IF_MISSING'] = '0'
os.environ['HYBRID_BOTH_FREEZE_MEMORY'] = '0'
os.environ['HYBRID_BOTH_SKIP_MEMORY_REBUILD'] = '0'


In [ ]:
!python scripts/build_hybrid_both_bundle.py --output-dir data/hybrid_both_bundle


In [ ]:
# Optional but recommended before the real run.
# Set HYBRID_BOTH_COMPUTE_XTB_IF_MISSING=1 in the train cell instead if you want lazy cache fills.
!python scripts/precompute_hybrid_both_xtb.py --bundle "$HYBRID_BOTH_BUNDLE" --cache-dir "$HYBRID_BOTH_XTB_CACHE"


In [ ]:
!python scripts/smoke_test_hybrid_both.py --bundle "$HYBRID_BOTH_BUNDLE" --xtb-cache-dir "$HYBRID_BOTH_XTB_CACHE"


In [ ]:
cmd = [
    'python', 'scripts/train_hybrid_both.py',
    '--bundle', os.environ['HYBRID_BOTH_BUNDLE'],
    '--epochs', os.environ['HYBRID_BOTH_EPOCHS'],
    '--batch-size', os.environ['HYBRID_BOTH_BATCH_SIZE'],
    '--learning-rate', os.environ['HYBRID_BOTH_LR'],
    '--weight-decay', os.environ['HYBRID_BOTH_WD'],
    '--xtb-cache-dir', os.environ['HYBRID_BOTH_XTB_CACHE'],
    '--output-dir', os.environ['HYBRID_BOTH_OUTPUT_DIR'],
    '--artifact-dir', os.environ['HYBRID_BOTH_ARTIFACT_DIR'],
    '--auto-resume-latest',
]
if os.environ.get('HYBRID_BOTH_LIMIT', '0') != '0':
    cmd += ['--limit', os.environ['HYBRID_BOTH_LIMIT']]
if os.environ.get('HYBRID_BOTH_COMPUTE_XTB_IF_MISSING', '0') == '1':
    cmd.append('--compute-xtb-if-missing')
if os.environ.get('HYBRID_BOTH_FREEZE_MEMORY', '0') == '1':
    cmd.append('--freeze-nexus-memory')
if os.environ.get('HYBRID_BOTH_SKIP_MEMORY_REBUILD', '0') == '1':
    cmd.append('--skip-nexus-memory-rebuild')

print(' '.join(cmd))
subprocess.check_call(cmd)
